In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
import my_data_manager as mdm

cat = mdm.load_cat("galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)


/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/mnt/c/Users/gasep/OneDrive/Documentos/GitHub/ClusteringComparison/ClusteringComparison/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [3]:
print(clusters[0].columns)
print(len(clusters))

Index(['galaxyId', 'haloId', 'pos_x', 'pos_y', 'pos_z', 'vel_x', 'vel_y',
       'vel_z', 'redshift', 'snapshot', 'sfr', 'm_star', 'm_coldgas',
       'Z_coldgas', 'R_coldgas', 'sfh_bin', 'sfh_nbin', 'RA', 'DEC', 'z_geo',
       'd_comoving', 'z_app', 'mag_u', 'mag_g', 'mag_r', 'mag_i', 'mag_z',
       'z_phot', 'firstHaloInFOFGroupId', 'log(m_200)'],
      dtype='object', name=0)
731


In [4]:
clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

In [5]:
import pandas as pd

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

In [ ]:
cluster_samples = {
    "raw": clusters,
    "clean": clean_clusters,
    "splus": splus_clusters
}

## Clustering

In [8]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


algorithms['GMM'] = clustering.run_GMM
algorithms['DBSCAN'] = clustering.run_DBSCAN
algorithms['HDBSCAN'] = clustering.run_HDBSCAN
algorithms['Optics'] = clustering.run_OPTICS
algorithms['Kmeans'] = clustering.run_Kmeans
algorithms['Aglomerative'] = clustering.run_Aglomerative_Clustering
algorithms['Affinity'] = clustering.run_Affinity_Propagation

params = {
    "max_clusters" : 0.2,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "max_iter" : 1000
}

In [ ]:
import numpy as np
import time

def ml_worker(alg, cluster_samples):

    predictions = []
    total_time = 0

    for sample in cluster_samples:
    
        X_data = sample["RA"].values
        Y_data = sample["DEC"].values
        
        data = np.column_stack((X_data, Y_data))
        
        data = np.asarray(data, dtype=np.float64)

        start_time = time.perf_counter()
        labels, probs, c = algorithms[alg](data, params)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        fof_id = np.int64(float(sample["firstHaloInFOFGroupId"].iloc[0]))
        labels = np.insert(labels, 0, fof_id)

        predictions.append(labels)
    
    return total_time, predictions

In [ ]:
from ds_plus import milaDS
import astro_utils as au
import numpy as np
import time
from astropy.stats import biweight_location

def dsp_worker(cluster_samples):

    predictions = []
    total_time = 0

    for sample in cluster_samples:
        
        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)
        
        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)

        X_Kpc, Y_Kpc = au.gal_Mpc_coords(X_data, Y_data, Z_data, x_cluster, y_cluster)
        X_Kpc *= 1000
        Y_Kpc *= 1000
        
        V_data = au.los_vel(Z_data, Z_clus)

        start_time = time.perf_counter()
        galaxy_info, grouping, summary = milaDS.DSp_groups(X_data, Y_data, V_data, Z_clus)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time
        
        labels = np.array([row[8] for row in grouping]) #9th column corresponds to the group

        fof_id = np.int64(float(sample["firstHaloInFOFGroupId"].iloc[0]))
        labels = np.insert(labels, 0, fof_id)

        predictions.append(labels)

    return delta_time, predictions



In [ ]:
from calsagos import lagasu
from calsagos import utils
from calsagos import clumberi
from astropy.stats import biweight_location
import numpy as np
import time

from IPython.display import clear_output

def calsagos_worker(cluster_samples):
    #- S-PLUS mock cosmology
    H_mock = 67.3
    Omega_L_mock = 0.685
    Omega_m_mock = 0.315

    range_cut_percentage = 0.2
    #-- GENERAL PARAMETERS
    n_galaxies = 4 # -- number of minimum of galaxies that a group or substructure must have

    predictions = []
    total_time = 0

    for sample in cluster_samples:

        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)

        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)
        
        cluster_mass = float(sample["log(m_200)"].iloc[0])
        
        range_cuts = int(len(sample)*range_cut_percentage)
        
        start_time = time.perf_counter()
        id_galaxy = indexes = np.arange(len(sample) + 1)
        
        #-- defining the cluster radius
        r200_kpc = utils.calc_radius_finn(cluster_mass, Z_clus, H_mock, Omega_L_mock, Omega_m_mock, "kiloparsec")

        #-- converting the radius in kpc to a radius in angular units
        #-- NOTE: if the user has an estimate of the r200 of the cluster it is not necessary to calculate this quantity
        r200_degree = utils.convert_kpc_to_angular_distance(r200_kpc, Z_clus, H_mock, Omega_m_mock, "degrees") 

        #-- select cluster members
        cluster_members = clumberi.clumberi(id_galaxy, X_data, Y_data, Z_data, Z_clus, x_cluster, y_cluster, range_cuts)

        # -- defining output parameters from clumberi
        id_member = cluster_members[0]
        ra_member = cluster_members[1]
        dec_member = cluster_members[2]
        redshift_member = cluster_members[3]


        #-- estimating the galaxy separation of galaxies in the cluster sample to be used as input in lagasu
        knn_distance = utils.calc_knn_galaxy_distance(ra_member, dec_member, n_galaxies)

        #-- determining the distance to the k-nearest neighbor of each galaxy in the cluster
        knn_galaxy_distance = knn_distance[0]

        try:
            typical_separation = utils.best_eps_dbscan(id_member, knn_galaxy_distance)
        except:
            clear_output(wait=True)
            print("Error ")
            print(id_member)
            print(len(id_member))
            print(knn_distance)
            print(len(knn_distance))

        #-- Assign galaxies to each substructures
        label_candidates = lagasu.lagasu(id_galaxy, X_data, Y_data, Z_data, 
                            range_cuts, typical_separation, n_galaxies, 'euclidean', 'dbscan', 
                            x_cluster, y_cluster, Z_clus, 
                            r200_degree, 'zspec')
        
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        #-- defining output parameters from lagasu
        id_candidates = label_candidates[0]
        ra_candidates = label_candidates[1]
        dec_candidates = label_candidates[2]
        redshift_candidates = label_candidates[3]
        label_zcut = label_candidates[4]
        label_final = label_candidates[5]    
    
        
        fof_id = np.int64(float(sample["firstHaloInFOFGroupId"].iloc[0]))
        label_final = np.insert(label_final, 0, fof_id)

        predictions.append(label_final)

    return total_time, predictions

In [12]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_ml_worker(worker, alg, thread_count, iterations):
    results = []
    
    with ProcessPoolExecutor(max_workers=thread_count) as executor:
            futures = [executor.submit(worker, alg) for _ in range(iterations)]
            for future in as_completed(futures):
                results.append(future.result())
    
    return results

In [13]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_worker(worker, thread_count, iterations):
    results = []

    with ProcessPoolExecutor(max_workers=thread_count) as executor:
        futures = [executor.submit(worker) for _ in range(iterations)]

        for future in as_completed(futures):
            results.append(future.result())
    
    return results

In [14]:
import openpyxl
from openpyxl import Workbook
from itertools import zip_longest

def save_xlsx(results, title):
    wb = Workbook()

    sheet1 = wb.active
    sheet1.title = "Execution Times"
    sheet1.append(["Execution Time (s)"])

    for duration, _ in results:
        sheet1.append([round(duration, 3)])

    for idx, (duration, data) in enumerate(results):
        sheet = wb.create_sheet(title=f"Result_{idx + 1}")

        for row in zip_longest(*data, fillvalue=""):
            sheet.append(row)

    # Save Excel file
    wb.save(f"{title}.xlsx")

In [ ]:
threads = 4
iterations = 10

for cs in cluster_samples.keys():
    cluster_sample = cluster_samples[cs]

    folder = f"results_{cs}_samples"
    os.makedirs(folder, exist_ok=True)

    for alg in algorithms.keys():
        ml_results = run_ml_worker(ml_worker, alg, threads, iterations)
        save_xlsx(ml_results, os.path.join(folder,f"ML_{alg}"))

    dsp_results = run_worker(dsp_worker, threads, iterations)
    save_xlsx(dsp_results, os.path.join(folder,f"DSP"))

    calsagos_results = run_worker(calsagos_worker, threads, iterations)
    save_xlsx(calsagos_results, os.path.join(folder,f"CALSAGOS"))